# 从零到项目上线

本章以 `Zesay` 为例，搭建一个前后端分离的全栈项目。

- 前端二选一：**Next.js** 或 **React + Vite**
- 后端：**FastAPI + uv**
- 版本管理：**Git**

## 1. 准备环境

安装 Node.js、Python、uv 和 Git，然后检查版本：

```powershell
node --version
npm --version
python --version
uv --version
git --version
```

## 2. 创建项目根目录并初始化 Git

```powershell
mkdir Zesay
cd Zesay
mkdir backend
mkdir docs
git init   ## 在根目录初始化 Git
New-Item README.md, AGENTS.md, .gitignore -ItemType File
```

frontend 在前端初始化时创建

最终结构：

```text
Zesay/
├─ backend/        - 后端代码
├─ frontend/       - 前端代码（从原型图文件中复制并完善）
├─ docs/MVP/       - 需求、设计和接口文档，用户故事与验收标准文档，原型HTML
├─ tests/MVP/      - 可运行测试代码
├─ README.md       - 给人看的
├─ .gitignore
└─ AGENTS.md       - 给 AI Agent 看的
```

`README.md` 至少应说明项目用途、技术栈、安装步骤、启动命令、环境变量、测试和部署方式。

`AGENTS.md` 可约定目录职责、代码风格、测试要求、`/api/` 路径规范，以及禁止提交密钥等规则。

## 3A. 前端方案一：Next.js（推荐）

在 `Zesay/` 根目录执行：

```powershell
npx create-next-app@latest frontend
```

交互选项建议：

```text
TypeScript                             Yes
ESLint                                 Yes
React Compiler                         Yes（如果出现）
Tailwind CSS                           Yes
代码放入 src/ 目录                      Yes
App Router                             Yes
Turbopack                              Yes（如果出现）
Customize the default import alias?    No
```

预览前端页面

```powershell
cd frontend
npm run dev
```

默认地址通常是 <http://localhost:3000>。

## 3B. 前端方案二：React + Vite

在 `Zesay/` 根目录执行：

```powershell
# 选用项目模板 react-ts
npm create vite@latest frontend -- --template react-ts
cd frontend
npm install
npm run dev
```

默认地址通常是 <http://localhost:5173>。

## 4. 创建 FastAPI + uv 后端

回到 `Zesay/` 根目录执行：

```powershell
cd backend
uv init
uv add "fastapi[standard]"
```

`uv init` 只初始化 Python 项目；还必须安装 FastAPI 和 Uvicorn。三者分工如下：

- `uv`：管理 Python、虚拟环境和依赖
- `FastAPI`：编写 Web API
- `Uvicorn`：运行 FastAPI 应用

在`main.py`复制下面的代码：

```python
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="Zesay API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000", "http://localhost:5173"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/api/health")
def health_check() -> dict[str, str]:
    return {"status": "ok"}
```

开发模式启动后端：

```powershell
uv run fastapi dev
```

- 健康检查：<http://127.0.0.1:8000/api/health>
- Swagger 文档：<http://127.0.0.1:8000/docs>

## 5. 前后端联调

分别打开两个终端。

终端一：

```powershell
cd Zesay/backend
uv run fastapi dev
```

终端二：

```powershell
cd Zesay/frontend
npm run dev
```

浏览器打开前端页面，打开开发者工具，切换到 Console（控制台）
输入下面代码，发送前端测试请求：

```ts
const response = await fetch("http://127.0.0.1:8000/api/health");
const data = await response.json();
console.log(data.status); //输出 ok 表示联调成功
```

## 6. 环境变量

在 frontend 目录下新建`.env.local`配置后端地址

### next.js框架：

```dotenv
NEXT_PUBLIC_API_URL=http://127.0.0.1:8000
```

导入方式：

1.在`app/page.tsx`的最上面添加

```TypeScript
"use client";
```

2.在`export default function Home() { `里面添加

```TypeScript
const apiUrl = process.env.NEXT_PUBLIC_API_URL;
console.log(apiUrl);
```

### Vite 构建的 React 框架：

```dotenv
VITE_API_URL=http://127.0.0.1:8000
```

导入方式：

在`App.tsx`文件的`const [count, setCount] = useState(0)`后面加上

```TypeScript
const apiUrl = import.meta.env.VITE_API_URL
console.log(apiUrl)
```

> `NEXT_PUBLIC_` 和 `VITE_` 开头的变量会暴露给浏览器，不能存放 API Key、数据库密码等秘密。

## 7. .gitignore

在项目根目录下新建`.gitignore`文件写入：

```gitignore
# Frontend
frontend/node_modules/
frontend/.next/
frontend/dist/
frontend/out/

# Backend
backend/.venv/
__pycache__/
.pytest_cache/
*.py[cod]

# Environment files 环境配置
.env
.env.*
!.env.example

# IDE and OS 编辑器和操作系统生成的文件
.vscode/
.idea/
.DS_Store
Thumbs.db
```

## 8. 检查并提交

前端检查（可选）：

```powershell
cd frontend
npm run lint    # 代码规范检查
npm run build   # 构建
```


提交代码：

```powershell
git status
git add .
git commit -m "项目初始化完成"
```